# ⚙️ Camada Silver – Limpeza e Padronização dos Dados Olist

A **camada Silver** é responsável por **limpar, padronizar e estruturar** os dados provenientes da camada Bronze, garantindo consistência e qualidade antes de serem usados em análises ou modelos na camada Gold.

**Regras gerais aplicadas neste notebook:**
- Todos os nomes de colunas em **português**
- Tipagem correta de todas as colunas
- Deduplicação com **Window Functions** (registro mais recente)
- Nenhuma alteração nas tabelas da camada Bronze

---

## 🏗️ Tabelas criadas neste notebook

| Origem (Bronze) | Destino (Silver) |
|:--|:--|
| tb_customers | silver.dim_consumidores |
| tb_orders | silver.fat_pedidos |
| tb_order_items | silver.fat_itens_pedidos |
| tb_order_payments | silver.fat_pagamentos_pedidos |
| tb_order_reviews | silver.fat_avaliacoes_pedidos |
| tb_products | silver.dim_produtos |
| tb_sellers | silver.dim_vendedores |
| tb_product_category_name_translation | silver.dim_categoria_produtos_traducao |
| tb_cotacao_dolar | silver.dim_cotacao_dolar |
| fat_pedidos + fat_pagamentos_pedidos + dim_cotacao_dolar | silver.fat_pedido_total |

## ⚙️ Configuração do Ambiente e Importações

In [0]:
# Importação de bibliotecas necessárias para transformações na camada Silver
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Definição do catálogo e schemas
catalogo = "medalhao"
bronze_db_name = "bronze"
silver_db_name = "silver"

print(f"Catálogo: {catalogo}")
print(f"Schema Bronze (origem): {bronze_db_name}")
print(f"Schema Silver (destino): {silver_db_name}")

Catálogo: medalhao
Schema Bronze (origem): bronze
Schema Silver (destino): silver


## 📖 Leitura das Tabelas Bronze

Carregamos todas as tabelas da camada Bronze que serão transformadas. As tabelas Bronze **não serão alteradas**.

In [0]:
# ---------------------------------------------------------------
# Leitura de todas as tabelas Bronze que serão transformadas
# As tabelas Bronze NÃO são modificadas — apenas lidas
# ---------------------------------------------------------------
customers_bronze_df = spark.table(f"{catalogo}.{bronze_db_name}.tb_customers")
orders_bronze_df = spark.table(f"{catalogo}.{bronze_db_name}.tb_orders")
order_items_bronze_df = spark.table(f"{catalogo}.{bronze_db_name}.tb_order_items")
order_payments_bronze_df = spark.table(f"{catalogo}.{bronze_db_name}.tb_order_payments")
order_reviews_bronze_df = spark.table(f"{catalogo}.{bronze_db_name}.tb_order_reviews")
products_bronze_df = spark.table(f"{catalogo}.{bronze_db_name}.tb_products")
sellers_bronze_df = spark.table(f"{catalogo}.{bronze_db_name}.tb_sellers")
category_translation_bronze_df = spark.table(f"{catalogo}.{bronze_db_name}.tb_product_category_name_translation")
cotacao_bronze_df = spark.table(f"{catalogo}.{bronze_db_name}.tb_cotacao_dolar")

print("✅ Todas as tabelas Bronze carregadas com sucesso!")

✅ Todas as tabelas Bronze carregadas com sucesso!


---
## 1️⃣ silver.dim_consumidores

**Origem:** `bronze.tb_customers`

**Transformações:**
- Renomeação de colunas para português
- Deduplicação por `id_consumidor` usando Window Function (mantém o registro mais recente por `timestamp_ingestion`)
- Estado e Cidade em **UPPER CASE**

In [0]:
# ---------------------------------------------------------------
# Tabela: bronze.tb_customers → silver.dim_consumidores
# ---------------------------------------------------------------

# Window Function para deduplicação sênior:
# Particiona por id_consumidor e ordena por timestamp_ingestion decrescente
# Garante que apenas o registro mais recente de cada consumidor seja mantido
w_consumidores = Window.partitionBy("customer_id").orderBy(F.col("timestamp_ingestion").desc())

dim_consumidores_df = (
    customers_bronze_df
    # Adiciona número de linha para identificar o registro mais recente por consumidor
    .withColumn("rn", F.row_number().over(w_consumidores))
    # Mantém apenas o registro mais recente (rn = 1)
    .filter(F.col("rn") == 1)
    .select(
        F.col("customer_id").alias("id_consumidor"),
        F.col("customer_unique_id").alias("id_consumidor_unico"),
        F.col("customer_zip_code_prefix").cast("string").alias("prefixo_cep"),
        # Cidade e Estado em UPPER CASE conforme regra de negócio
        F.upper(F.trim(F.col("customer_city"))).alias("cidade"),
        F.upper(F.trim(F.col("customer_state"))).alias("estado")
    )
)

# Escrita na camada Silver
(
    dim_consumidores_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{silver_db_name}.dim_consumidores")
)

print(f"✅ Tabela silver.dim_consumidores criada com sucesso! ({dim_consumidores_df.count()} registros)")

✅ Tabela silver.dim_consumidores criada com sucesso! (99441 registros)


---
## 2️⃣ silver.fat_pedidos

**Origem:** `bronze.tb_orders`

**Transformações:**
- Tradução do status do pedido de inglês para português
- Criação de colunas derivadas de tempo de entrega
- Indicador `entrega_no_prazo` (Sim/Não/Não Entregue)

In [0]:
# ---------------------------------------------------------------
# Tabela: bronze.tb_orders → silver.fat_pedidos
# ---------------------------------------------------------------

fat_pedidos_df = (
    orders_bronze_df
    .select(
        F.col("order_id").alias("id_pedido"),
        F.col("customer_id").alias("id_consumidor"),

        # Tradução do status do pedido de inglês para português
        # Regra de negócio: mapeamento explícito de todos os status possíveis
        F.when(F.col("order_status") == "delivered", "entregue")
         .when(F.col("order_status") == "canceled", "cancelado")
         .when(F.col("order_status") == "shipped", "enviado")
         .when(F.col("order_status") == "processing", "processando")
         .when(F.col("order_status") == "invoiced", "faturado")
         .when(F.col("order_status") == "unavailable", "indisponivel")
         .when(F.col("order_status") == "created", "criado")
         .when(F.col("order_status") == "approved", "aprovado")
         .otherwise(F.col("order_status"))
         .alias("status"),

        # Conversão das colunas de data para o tipo timestamp
        F.to_timestamp("order_purchase_timestamp").alias("data_pedido"),
        F.to_timestamp("order_approved_at").alias("data_aprovacao"),
        F.to_timestamp("order_delivered_carrier_date").alias("data_entrega_transportadora"),
        F.to_timestamp("order_delivered_customer_date").alias("data_entrega_cliente"),
        F.to_timestamp("order_estimated_delivery_date").alias("data_entrega_estimada")
    )
    # Criação de colunas derivadas de análise de tempo de entrega
    .withColumn(
        "tempo_entrega_dias",
        F.datediff(F.col("data_entrega_cliente"), F.col("data_pedido"))
    )
    .withColumn(
        "tempo_entrega_estimado_dias",
        F.datediff(F.col("data_entrega_estimada"), F.col("data_pedido"))
    )
    # Diferença entre tempo real e estimado: positivo = atrasado, negativo = adiantado
    .withColumn(
        "diferenca_entrega_dias",
        F.col("tempo_entrega_dias") - F.col("tempo_entrega_estimado_dias")
    )
    # Indicador textual de pontualidade:
    # 'Sim' se entregue no prazo, 'Não' se atrasado, 'Não Entregue' para demais status
    .withColumn(
        "entrega_no_prazo",
        F.when(
            (F.col("status") == "entregue") & (F.col("diferenca_entrega_dias") <= 0),
            "Sim"
        ).when(
            (F.col("status") == "entregue") & (F.col("diferenca_entrega_dias") > 0),
            "Não"
        ).otherwise("Não Entregue")
    )
)

(
    fat_pedidos_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{silver_db_name}.fat_pedidos")
)

print(f"✅ Tabela silver.fat_pedidos criada com sucesso! ({fat_pedidos_df.count()} registros)")

✅ Tabela silver.fat_pedidos criada com sucesso! (99441 registros)


---
## 3️⃣ silver.fat_itens_pedidos

**Origem:** `bronze.tb_order_items`

**Transformações:**
- Renomeação de colunas para português
- Tipagem correta (preços como double)

In [0]:
# ---------------------------------------------------------------
# Tabela: bronze.tb_order_items → silver.fat_itens_pedidos
# ---------------------------------------------------------------

fat_itens_pedidos_df = (
    order_items_bronze_df
    .select(
        F.col("order_id").alias("id_pedido"),
        F.col("order_item_id").cast("int").alias("id_item"),
        F.col("product_id").alias("id_produto"),
        F.col("seller_id").alias("id_vendedor"),
        F.to_timestamp("shipping_limit_date").alias("data_limite_envio"),
        # Preço em BRL — será usado para calcular o valor em USD na camada Gold
        F.col("price").cast("double").alias("preco_BRL"),
        F.col("freight_value").cast("double").alias("preco_frete")
    )
)

(
    fat_itens_pedidos_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{silver_db_name}.fat_itens_pedidos")
)

print(f"✅ Tabela silver.fat_itens_pedidos criada com sucesso! ({fat_itens_pedidos_df.count()} registros)")

✅ Tabela silver.fat_itens_pedidos criada com sucesso! (112650 registros)


---
## 4️⃣ silver.fat_pagamentos_pedidos

**Origem:** `bronze.tb_order_payments`

**Transformações:**
- Tradução do tipo de pagamento de inglês para português

In [0]:
# ---------------------------------------------------------------
# Tabela: bronze.tb_order_payments → silver.fat_pagamentos_pedidos
# ---------------------------------------------------------------

fat_pagamentos_pedidos_df = (
    order_payments_bronze_df
    .select(
        F.col("order_id").alias("id_pedido"),
        F.col("payment_sequential").cast("int").alias("sequencia_pagamento"),

        # Tradução do tipo de pagamento de inglês para português
        # Regra de negócio: mapeamento explícito de todos os métodos disponíveis
        F.when(F.col("payment_type") == "credit_card", "Cartão de Crédito")
         .when(F.col("payment_type") == "boleto", "Boleto")
         .when(F.col("payment_type") == "voucher", "Voucher")
         .when(F.col("payment_type") == "debit_card", "Cartão de Débito")
         .when(F.col("payment_type") == "not_defined", "Não Definido")
         .otherwise(F.col("payment_type"))
         .alias("tipo_pagamento"),

        F.col("payment_installments").cast("int").alias("parcelas"),
        F.col("payment_value").cast("double").alias("valor_pagamento")
    )
)

(
    fat_pagamentos_pedidos_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{silver_db_name}.fat_pagamentos_pedidos")
)

print(f"✅ Tabela silver.fat_pagamentos_pedidos criada com sucesso! ({fat_pagamentos_pedidos_df.count()} registros)")

✅ Tabela silver.fat_pagamentos_pedidos criada com sucesso! (103886 registros)


---
## 5️⃣ silver.fat_avaliacoes_pedidos

**Origem:** `bronze.tb_order_reviews`

**Transformações:**
- `try_to_timestamp` para tolerância a erros em colunas de data
- Remoção de registros com pedido inválido ou datas no futuro
- Preenchimento de nulos com "Sem comentário" e "Sem título"

In [0]:
# ---------------------------------------------------------------
# Tabela: bronze.tb_order_reviews → silver.fat_avaliacoes_pedidos
# ---------------------------------------------------------------

fat_avaliacoes_pedidos_df = (
    order_reviews_bronze_df
    .select(
        F.col("review_id").alias("id_avaliacao"),
        F.col("order_id").alias("id_pedido"),
        F.col("review_score").cast("int").alias("nota_avaliacao"),

        # Tratamento de nulos: preenche títulos vazios com string padrão
        F.when(
            F.col("review_comment_title").isNull() | (F.trim(F.col("review_comment_title")) == ""),
            "Sem título"
        ).otherwise(F.trim(F.col("review_comment_title")))
        .alias("titulo_avaliacao"),

        # Tratamento de nulos: preenche comentários vazios com string padrão
        F.when(
            F.col("review_comment_message").isNull() | (F.trim(F.col("review_comment_message")) == ""),
            "Sem comentário"
        ).otherwise(F.trim(F.col("review_comment_message")))
        .alias("comentario_avaliacao"),

        # Uso de try_to_timestamp para tolerância a falhas:
        # Evita que formatos inválidos de data quebrem o processamento
        F.try_to_timestamp(F.col("review_creation_date")).alias("data_criacao_avaliacao"),
        F.try_to_timestamp(F.col("review_answer_timestamp")).alias("data_resposta_avaliacao")
    )
    # Remove registros com pedido inválido (nulo)
    .filter(F.col("id_pedido").isNotNull())
    # Remove registros com datas de criação de comentário no futuro
    .filter(
        F.col("data_criacao_avaliacao").isNull() |
        (F.col("data_criacao_avaliacao") <= F.current_timestamp())
    )
)

(
    fat_avaliacoes_pedidos_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{silver_db_name}.fat_avaliacoes_pedidos")
)

print(f"✅ Tabela silver.fat_avaliacoes_pedidos criada com sucesso! ({fat_avaliacoes_pedidos_df.count()} registros)")

✅ Tabela silver.fat_avaliacoes_pedidos criada com sucesso! (99224 registros)


---
## 6️⃣ silver.dim_produtos

**Origem:** `bronze.tb_products`

**Transformações:**
- Deduplicação sênior por `id_produto` usando Window Function
- Renomeação e tipagem de todas as colunas

In [0]:
# ---------------------------------------------------------------
# Tabela: bronze.tb_products → silver.dim_produtos
# ---------------------------------------------------------------

# Window Function para deduplicação sênior por produto
# Mantém apenas o registro mais recente por id_produto
w_produtos = Window.partitionBy("product_id").orderBy(F.col("timestamp_ingestion").desc())

dim_produtos_df = (
    products_bronze_df
    .withColumn("rn", F.row_number().over(w_produtos))
    .filter(F.col("rn") == 1)
    .select(
        F.col("product_id").alias("id_produto"),
        F.col("product_category_name").alias("categoria_produto"),
        F.col("product_name_lenght").cast("int").alias("tamanho_nome_produto"),
        F.col("product_description_lenght").cast("int").alias("tamanho_descricao_produto"),
        F.col("product_photos_qty").cast("int").alias("quantidade_fotos"),
        F.col("product_weight_g").cast("double").alias("peso_produto_gramas"),
        F.col("product_length_cm").cast("double").alias("comprimento_centimetros"),
        F.col("product_height_cm").cast("double").alias("altura_centimetros"),
        F.col("product_width_cm").cast("double").alias("largura_centimetros")
    )
)

(
    dim_produtos_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{silver_db_name}.dim_produtos")
)

print(f"✅ Tabela silver.dim_produtos criada com sucesso! ({dim_produtos_df.count()} registros)")

✅ Tabela silver.dim_produtos criada com sucesso! (32951 registros)


---
## 7️⃣ silver.dim_vendedores

**Origem:** `bronze.tb_sellers`

**Transformações:**
- Deduplicação sênior por `id_vendedor` usando Window Function
- Estado e Cidade em UPPER CASE

In [0]:
# ---------------------------------------------------------------
# Tabela: bronze.tb_sellers → silver.dim_vendedores
# ---------------------------------------------------------------

# Window Function para deduplicação sênior por vendedor
w_vendedores = Window.partitionBy("seller_id").orderBy(F.col("timestamp_ingestion").desc())

dim_vendedores_df = (
    sellers_bronze_df
    .withColumn("rn", F.row_number().over(w_vendedores))
    .filter(F.col("rn") == 1)
    .select(
        F.col("seller_id").alias("id_vendedor"),
        F.col("seller_zip_code_prefix").cast("string").alias("prefixo_cep"),
        # Cidade e Estado em UPPER CASE conforme regra de negócio
        F.upper(F.trim(F.col("seller_city"))).alias("cidade"),
        F.upper(F.trim(F.col("seller_state"))).alias("estado")
    )
)

(
    dim_vendedores_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{silver_db_name}.dim_vendedores")
)

print(f"✅ Tabela silver.dim_vendedores criada com sucesso! ({dim_vendedores_df.count()} registros)")

✅ Tabela silver.dim_vendedores criada com sucesso! (3095 registros)


---
## 8️⃣ silver.dim_categoria_produtos_traducao

**Origem:** `bronze.tb_product_category_name_translation`

**Transformações:**
- Renomeação para manter nome em português e em inglês

In [0]:
# ---------------------------------------------------------------
# Tabela: bronze.tb_product_category_name_translation
#      → silver.dim_categoria_produtos_traducao
# ---------------------------------------------------------------

dim_categoria_traducao_df = (
    category_translation_bronze_df
    .select(
        # Nome original em português (fonte)
        F.col("product_category_name").alias("nome_produto_pt"),
        # Nome traduzido para inglês
        F.col("product_category_name_english").alias("nome_produto_en")
    )
    .filter(F.col("nome_produto_pt").isNotNull())
)

(
    dim_categoria_traducao_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{silver_db_name}.dim_categoria_produtos_traducao")
)

print(f"✅ Tabela silver.dim_categoria_produtos_traducao criada com sucesso! ({dim_categoria_traducao_df.count()} registros)")

✅ Tabela silver.dim_categoria_produtos_traducao criada com sucesso! (71 registros)


---
## 9️⃣ silver.dim_cotacao_dolar

**Origem:** `bronze.tb_cotacao_dolar`

**Transformações:**
- A API do BCB não fornece cotações para fins de semana
- Criamos um **calendário contínuo** cobrindo todos os dias do período
- Usamos `last(ignorenulls=True)` sobre Window Function para preencher os finais de semana com a cotação de fechamento da **sexta-feira anterior**

In [0]:
# ---------------------------------------------------------------
# Tabela: bronze.tb_cotacao_dolar → silver.dim_cotacao_dolar
# ---------------------------------------------------------------

# Preparação da tabela de cotação com data extraída do timestamp
cotacao_df = (
    cotacao_bronze_df
    .select(
        F.to_date(F.col("dataHoraCotacao")).alias("data"),
        F.col("cotacaoCompra").cast("double").alias("cotacao_dolar")
    )
    # Agrega por data mantendo a última cotação do dia
    .groupBy("data")
    .agg(F.last("cotacao_dolar").alias("cotacao_dolar"))
)

# Determina as datas de início e fim do período de cotações
data_min = cotacao_df.agg(F.min("data")).collect()[0][0]
data_max = cotacao_df.agg(F.max("data")).collect()[0][0]

# Cria um calendário contínuo com todos os dias do período
# Isso é necessário porque a API do BCB não retorna fins de semana
calendario_df = (
    spark.sql(f"""
        SELECT explode(sequence(
            to_date('{data_min}'),
            to_date('{data_max}'),
            interval 1 day
        )) AS data
    """)
)

# Faz left join do calendário com as cotações para criar lacunas nos fins de semana
cotacao_completa_df = calendario_df.join(cotacao_df, on="data", how="left")

# Window Function para preencher lacunas de fim de semana:
# last(ignorenulls=True) busca o último valor não nulo anterior na ordem cronológica
# Isso substitui o fim de semana pela cotação de fechamento da sexta-feira anterior
w_cotacao = Window.orderBy("data").rowsBetween(Window.unboundedPreceding, Window.currentRow)

dim_cotacao_dolar_df = (
    cotacao_completa_df
    .withColumn(
        "cotacao_dolar",
        F.last(F.col("cotacao_dolar"), ignorenulls=True).over(w_cotacao)
    )
)

(
    dim_cotacao_dolar_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{silver_db_name}.dim_cotacao_dolar")
)

print(f"✅ Tabela silver.dim_cotacao_dolar criada com sucesso! ({dim_cotacao_dolar_df.count()} registros)")

/databricks/python/lib/python3.10/site-packages/pyspark/sql/connect/expressions.py:968: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✅ Tabela silver.dim_cotacao_dolar criada com sucesso! (729 registros)


---
## 🔟 silver.fat_pedido_total

**Tabela consolidada final da camada Silver**

Une `fat_pedidos` + `fat_pagamentos_pedidos` + `dim_cotacao_dolar` para calcular o valor total pago em BRL e em USD.

Valores financeiros são **arredondados para 2 casas decimais**.

In [0]:
# ---------------------------------------------------------------
# Tabela Final Silver: fat_pedido_total
# Une fat_pedidos + fat_pagamentos_pedidos + dim_cotacao_dolar
# ---------------------------------------------------------------

# Leitura das tabelas Silver recém criadas
fat_pedidos_silver = spark.table(f"{catalogo}.{silver_db_name}.fat_pedidos")
fat_pagamentos_silver = spark.table(f"{catalogo}.{silver_db_name}.fat_pagamentos_pedidos")
dim_cotacao_silver = spark.table(f"{catalogo}.{silver_db_name}.dim_cotacao_dolar")

# Agrega pagamentos por pedido para obter o valor total pago em BRL
pagamentos_agg = (
    fat_pagamentos_silver
    .groupBy("id_pedido")
    .agg(F.sum("valor_pagamento").alias("valor_total_pago_brl"))
)

# Join de pedidos com pagamentos agregados
pedidos_com_pagamento = fat_pedidos_silver.join(pagamentos_agg, on="id_pedido", how="left")

# Join com cotação do dólar pela data do pedido
# Extrai a data (sem hora) para fazer o join com a tabela de cotação
fat_pedido_total_df = (
    pedidos_com_pagamento
    .withColumn("data_pedido_dt", F.to_date(F.col("data_pedido")))
    .join(
        dim_cotacao_silver.select("data", "cotacao_dolar"),
        pedidos_com_pagamento["data_pedido"].cast("date") == dim_cotacao_silver["data"],
        how="left"
    )
    .select(
        F.col("id_pedido"),
        F.col("id_consumidor"),
        F.col("status"),
        # Arredondamento obrigatório para 2 casas decimais
        F.round(F.col("valor_total_pago_brl"), 2).alias("valor_total_pago_brl"),
        # Conversão BRL → USD usando a cotação do dia do pedido
        F.round(F.col("valor_total_pago_brl") / F.col("cotacao_dolar"), 2).alias("valor_total_pago_usd"),
        F.col("data_pedido")
    )
)

(
    fat_pedido_total_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{silver_db_name}.fat_pedido_total")
)

print(f"✅ Tabela silver.fat_pedido_total criada com sucesso! ({fat_pedido_total_df.count()} registros)")

✅ Tabela silver.fat_pedido_total criada com sucesso! (99441 registros)


---
## 🚀 Otimização Física – Delta Optimize com ZORDER

O comando `OPTIMIZE` compacta os arquivos Delta Lake, e o `ZORDER BY` co-localiza dados relacionados nos mesmos arquivos físicos, melhorando significativamente a performance de consultas analíticas que filtram por `id_pedido` e `data_pedido`.

In [0]:
# ---------------------------------------------------------------
# Otimização física das principais tabelas fato com Delta Optimize
# ZORDER BY co-localiza dados nas mesmas partições para acelerar consultas
# ---------------------------------------------------------------

print("🔧 Iniciando otimização das tabelas fato...")

# Otimiza a tabela de pedidos indexando por id_pedido e data_pedido
spark.sql(f"""
    OPTIMIZE {catalogo}.{silver_db_name}.fat_pedidos
    ZORDER BY (id_pedido, data_pedido)
""")
print("✅ fat_pedidos otimizado!")

# Otimiza a tabela de itens de pedidos
spark.sql(f"""
    OPTIMIZE {catalogo}.{silver_db_name}.fat_itens_pedidos
    ZORDER BY (id_pedido)
""")
print("✅ fat_itens_pedidos otimizado!")

# Otimiza a tabela de pagamentos
spark.sql(f"""
    OPTIMIZE {catalogo}.{silver_db_name}.fat_pagamentos_pedidos
    ZORDER BY (id_pedido)
""")
print("✅ fat_pagamentos_pedidos otimizado!")

# Otimiza a tabela consolidada final
spark.sql(f"""
    OPTIMIZE {catalogo}.{silver_db_name}.fat_pedido_total
    ZORDER BY (id_pedido, data_pedido)
""")
print("✅ fat_pedido_total otimizado!")

print("\n🎉 Notebook da Camada Silver finalizado com sucesso!")

🔧 Iniciando otimização das tabelas fato...
✅ fat_pedidos otimizado!
✅ fat_itens_pedidos otimizado!
✅ fat_pagamentos_pedidos otimizado!
✅ fat_pedido_total otimizado!

🎉 Notebook da Camada Silver finalizado com sucesso!


## ✅ Verificação Final – Tabelas Silver

In [0]:
# ---------------------------------------------------------------
# Verificação final: lista todas as tabelas criadas no schema Silver
# ---------------------------------------------------------------
print("📋 Tabelas criadas na camada Silver:")
spark.sql(f"SHOW TABLES IN {catalogo}.{silver_db_name}").display()

📋 Tabelas criadas na camada Silver:


database,tableName,isTemporary
silver,dim_categoria_produtos_traducao,false
silver,dim_consumidores,false
silver,dim_cotacao_dolar,false
silver,dim_produtos,false
silver,dim_vendedores,false
silver,fat_avaliacoes_pedidos,false
silver,fat_itens_pedidos,false
silver,fat_pagamentos_pedidos,false
silver,fat_pedido_total,false
silver,fat_pedidos,false
